# Bayesian Optimization Capstone -- Week 13: FINAL ROUND

All changes from Week 12 are marked with `# CHANGED:` or `# NEW:` comments.

### Summary of changes from Week 12
| Change | Week 12 | Week 13 | Reason |
|---|---|---|---|
| F1 sweep coordinate | W12: [0.2276, 0.9500] | W13: [0.5918, 0.2396] | Final planned sweep point -- completes 4-point systematic grid |
| kappa for F2 | 0.5 | 0.5 | Unchanged -- close to best 0.678, final exploitation |
| kappa for F3 | 0.5 | 0.5 | Unchanged -- near best, final exploitation |
| kappa for F4 | 2.0 | 1.0 | Moderately exploit near R10 basin (best 0.520) |
| kappa for F5 | 2.0 | 1.0 | Moderate exploitation toward best 6641 region |
| kappa for F6 | 2.5 | 2.5 | Unchanged -- final exploratory shot |
| kappa for F7 | 0.5 | 0.5 | Unchanged -- exploit near R11 best 1.972 |
| kappa for F8 | 2.0 | 1.5 | Balanced final shot after W12 regression |
| Week 12 data appended | No | Yes | Final dataset |
| Seed base | 222 | 333 | Final seed rotation |

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist

# CHANGED: final seed base 333
SEED_BASE = 333
np.random.seed(SEED_BASE)
print('Libraries loaded -- FINAL ROUND')

## 2. Load Complete Dataset (Initial + All 12 Query Rounds)

**CHANGED from Week 12:** Week 12 queries and outputs appended. This is the full final dataset.

In [ ]:
DATA_PATH = './data/'

week1_queries={1:np.array([0.020584,0.969910]),2:np.array([0.813958,0.968718]),3:np.array([0.412118,0.126161,0.493019]),4:np.array([0.390847,0.455494,0.413276,0.470071]),5:np.array([0.342735,0.792788,0.998424,0.935381]),6:np.array([0.690974,0.164681,0.103053,0.037619,0.154198]),7:np.array([0.030065,0.557667,0.280418,0.193751,0.310746,0.755561]),8:np.array([0.190497,0.131345,0.026522,0.042568,0.730959,0.283019,0.005482,0.388835])}
week1_outputs={1:1.966e-321,2:-0.017659539446735338,3:-0.03594378093624019,4:-1.3242014729441993,5:2021.562348501267,6:-1.8227163856485316,7:1.3096626182809736,8:9.810010198309}
week2_queries={1:np.array([0.999999,0.999999]),2:np.array([0.673666,0.918591]),3:np.array([0.506031,0.565092,0.511916]),4:np.array([0.462210,0.470559,0.312877,0.405903]),5:np.array([0.505785,0.709925,0.994838,0.985391]),6:np.array([0.506031,0.565092,0.511916,0.972186,0.614903]),7:np.array([0.057378,0.366665,0.383868,0.136267,0.331203,0.752374]),8:np.array([0.032926,0.283604,0.006343,0.409730,0.814350,0.536043,0.151336,0.699855])}
week2_outputs={1:1.517648729565899e-192,2:0.510292287604992,3:-0.010612319162868315,4:-2.197214238202132,5:2201.3475945869486,6:-1.107214322524676,7:1.9584243277849602,8:9.8526837633915}
week3_queries={1:np.array([0.418744,0.463065]),2:np.array([0.718071,0.879689]),3:np.array([0.611094,0.382817,0.600071]),4:np.array([0.424666,0.520010,0.477146,0.381999]),5:np.array([0.524778,0.842229,0.984007,0.984075]),6:np.array([0.357049,0.223854,0.497338,0.808843,0.076793]),7:np.array([0.007196,0.287182,0.431297,0.103298,0.259892,0.771925]),8:np.array([0.024544,0.175956,0.116596,0.359046,0.449942,0.482790,0.135292,0.369405])}
week3_outputs={1:-0.005743923859916853,2:0.43088672102497183,3:-0.06351273264725116,4:-2.2609959093621828,5:2761.25406610152,6:-0.42557686218603197,7:1.7793279687424954,8:9.8684416967155}
week4_queries={1:np.array([0.050000,0.683166]),2:np.array([0.684422,0.983396]),3:np.array([0.773956,0.438878,0.858598]),4:np.array([0.437029,0.343930,0.415384,0.380827]),5:np.array([0.706600,0.999024,0.993655,0.986456]),6:np.array([0.400742,0.367525,0.463774,0.773392,0.094658]),7:np.array([0.085756,0.416199,0.486065,0.134693,0.401951,0.908800]),8:np.array([0.139770,0.331568,0.075811,0.049482,0.626413,0.402617,0.343854,0.516662])}
week4_outputs={1:1.1211575454297984e-144,2:0.45409752904570855,3:-0.03246674358598379,4:0.3657242066980797,5:5079.038688784203,6:-0.21379268800966483,7:1.178017048841175,8:9.8783244032591}
week5_queries={1:np.array([0.669598,0.050000]),2:np.array([0.735047,0.098026]),3:np.array([0.998632,0.651661,0.940995]),4:np.array([0.439093,0.344645,0.450032,0.381672]),5:np.array([0.934309,0.964839,0.991679,0.961572]),6:np.array([0.335944,0.499225,0.420450,0.716945,0.053954]),7:np.array([0.682352,0.053821,0.220360,0.184372,0.175906,0.812095]),8:np.array([0.185222,0.323353,0.012446,0.210805,0.688551,0.466240,0.073074,0.434930])}
week5_outputs={1:1.2526182543541078e-139,2:0.37763279019568097,3:-0.17975723176997382,4:-0.10027419641986723,5:6204.005487139041,6:-0.562862619219461,7:0.8855931591908454,8:9.8679736752075}
week6_queries={1:np.array([0.050000,0.135930]),2:np.array([0.673903,0.198094]),3:np.array([0.941407,0.229430,0.767542]),4:np.array([0.467922,0.285493,0.372897,0.437788]),5:np.array([0.979491,0.966487,0.975465,0.931802]),6:np.array([0.469673,0.447655,0.124057,0.754850,0.972988]),7:np.array([0.062617,0.306301,0.403156,0.127209,0.385557,0.539932]),8:np.array([0.119949,0.027514,0.106943,0.240757,0.633207,0.420766,0.184013,0.620722])}
week6_outputs={1:-8.02263126364567e-154,2:0.5634168991004778,3:-0.1296499558422766,4:-1.8752576382494044,5:6210.693425379411,6:-1.4987906811967835,7:1.926382419159767,8:9.953627458159099}
week7_queries={1:np.array([0.660553,0.429899]),2:np.array([0.594914,0.176260]),3:np.array([0.696015,0.674845,0.656797]),4:np.array([0.346593,0.326325,0.369944,0.337553]),5:np.array([0.943597,0.976718,0.948137,0.999531]),6:np.array([0.398706,0.419380,0.631066,0.991715,0.106696]),7:np.array([0.098466,0.251391,0.280505,0.082460,0.342315,0.610119]),8:np.array([0.115531,0.115602,0.041889,0.228086,0.497790,0.351216,0.169864,0.523432])}
week7_outputs={1:-3.879375797615792e-30,2:0.02026092421605935,3:-0.10904856153213754,4:-0.045864326272869516,5:6445.985517684735,6:-0.3317516878378579,7:1.8604836309219728,8:9.8987415417546}
week8_queries={1:np.array([0.950000,0.050000]),2:np.array([0.721334,0.175922]),3:np.array([0.658767,0.908308,0.483438]),4:np.array([0.439619,0.350673,0.413110,0.312941]),5:np.array([0.996673,0.962839,0.859982,0.997910]),6:np.array([0.491437,0.403479,0.538615,0.795215,0.222769]),7:np.array([0.007696,0.240277,0.401756,0.062769,0.393763,0.711757]),8:np.array([0.057963,0.103880,0.204591,0.123209,0.685804,0.464601,0.114600,0.759717])}
week8_outputs={1:4.406064392151042e-291,2:0.4867931130901393,3:-0.0291843735673941,4:-0.36917659932470626,5:5933.001366157545,6:-0.4181033220567929,7:1.6440434279695768,8:9.9520187929201}
week9_queries={1:np.array([0.940955,0.791709]),2:np.array([0.704680,0.247092]),3:np.array([0.323377,0.991377,0.696409]),4:np.array([0.331686,0.403962,0.384508,0.366948]),5:np.array([0.968878,0.999777,0.923736,0.932342]),6:np.array([0.430444,0.425197,0.332502,0.969611,0.065639]),7:np.array([0.323377,0.991377,0.696409,0.663859,0.371318,0.547498]),8:np.array([0.121457,0.128220,0.144615,0.144979,0.670691,0.558397,0.143412,0.547020])}
week9_outputs={1:9.223582156551595e-87,2:0.6467241284185502,3:-0.15126420286652115,4:0.05050314378579346,5:5852.708262052304,6:-0.6297411638489756,7:0.11380431641114436,8:9.9794831109085}
week10_queries={1:np.array([0.227592,0.561706]),2:np.array([0.699383,0.228370]),3:np.array([0.374226,0.408388,0.484620]),4:np.array([0.368992,0.371946,0.424105,0.380723]),5:np.array([0.970967,0.967965,0.970716,0.974130]),6:np.array([0.396977,0.367153,0.935090,0.877802,0.012961]),7:np.array([0.189498,0.340978,0.372412,0.081751,0.276842,0.652574]),8:np.array([0.011253,0.328964,0.172920,0.266480,0.765727,0.661913,0.251360,0.533296])}
week10_outputs={1:1.2738177869706492e-41,2:0.6010397743985006,3:-0.01307504556329871,4:0.519690005368759,5:6640.5819888313845,6:-0.5223502766659631,7:0.07653735437639558,8:9.9006020766909}
week11_queries={1:np.array([0.272742,0.308863]),2:np.array([0.702030,0.317780]),3:np.array([0.444489,0.533421,0.465657]),4:np.array([0.443639,0.400977,0.372879,0.379218]),5:np.array([0.958439,0.974649,0.971165,0.935566]),6:np.array([0.153661,0.169303,0.505964,0.658119,0.767581]),7:np.array([0.016967,0.166633,0.285123,0.143395,0.368758,0.568264]),8:np.array([0.060189,0.015051,0.130993,0.029523,0.842393,0.800701,0.042999,0.794811])}
week11_outputs={1:-1.2971876943970279e-25,2:0.6781962056212831,3:-0.00015118518262546637,4:0.02967170003690045,5:6005.715406314301,6:-1.5131940553228198,7:1.9723819339089852,8:9.8196878350814}
week12_queries={1:np.array([0.227592,0.950000]),2:np.array([0.716930,0.291379]),3:np.array([0.463564,0.479712,0.457291]),4:np.array([0.373699,0.346974,0.389289,0.386809]),5:np.array([0.941375,0.935955,0.991898,0.986310]),6:np.array([0.661409,0.451123,0.500843,0.282856,0.749130]),7:np.array([0.018182,0.319854,0.259762,0.166866,0.413766,0.499636]),8:np.array([0.212508,0.087460,0.121616,0.604099,0.923389,0.929909,0.061673,0.004495])}
# CHANGED: Week 12 outputs added
week12_outputs={1:-5.566736041795097e-184,2:0.6649051053599078,3:-0.00799258549516528,4:0.24803750475223696,5:6271.127834411607,6:-1.5667325873820648,7:1.4257751672053562,8:9.498190362801}

all_week_queries = [week1_queries,week2_queries,week3_queries,week4_queries,week5_queries,
                    week6_queries,week7_queries,week8_queries,week9_queries,week10_queries,
                    week11_queries,week12_queries]
all_week_outputs = [week1_outputs,week2_outputs,week3_outputs,week4_outputs,week5_outputs,
                    week6_outputs,week7_outputs,week8_outputs,week9_outputs,week10_outputs,
                    week11_outputs,week12_outputs]

# CHANGED: full 12-round history
functions = {}
for i in range(1, 9):
    X = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    for wq, wo in zip(all_week_queries, all_week_outputs):
        X = np.vstack([X, wq[i]])
        y = np.append(y, wo[i])
    functions[i] = {'X': X, 'y': y}

print(f"{'Fn':<5} {'n pts':<8} {'All-time Best':<16} {'W12 Output':<16} {'W12 Improved?'}")
print('-' * 65)
for i in range(1, 9):
    prev = max(functions[i]['y'][:-1])
    improved = week12_outputs[i] > prev
    print(f"F{i:<4} {len(functions[i]['y']):<8} {functions[i]['y'].max():<16.6f} "
          f"{week12_outputs[i]:<16.6f} {'YES' if improved else 'no'}")

## 3. Final Feature-Output Correlation Summary

Complete correlation analysis across all 12 rounds of data.

In [ ]:
print("Final feature-output Pearson correlations (full dataset):\n")
for func_id, d in functions.items():
    X, y = d['X'], d['y']
    corrs = [round(float(np.corrcoef(X[:, j], y)[0, 1]), 3) for j in range(X.shape[1])]
    labels = [f'x{j+1}:{c}' for j, c in enumerate(corrs)]
    dominant = sorted(range(len(corrs)), key=lambda j: abs(corrs[j]), reverse=True)[:2]
    dominant_str = ', '.join(f'x{j+1}' for j in dominant)
    print(f"F{func_id}: {', '.join(labels)}  |  dominant: {dominant_str}")

## 4. Core GP and Acquisition Functions

Unchanged. Seed base 333. Per-function seed offset maintained.

In [ ]:
def fit_gp(X, y):
    gp = GaussianProcessRegressor(
        kernel=Matern(nu=2.5), n_restarts_optimizer=10, normalize_y=True
    )
    gp.fit(X, y)
    return gp

def suggest_next_query(X, y, n_dims, n_candidates=100000, kappa=2.0, seed=333):
    """CHANGED: seed base updated to 333."""
    rng = np.random.default_rng(seed)
    gp = fit_gp(X, y)
    candidates = rng.uniform(0, 1, size=(n_candidates, n_dims))
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + kappa * sigma
    best_idx = np.argmax(ucb)
    best_x = candidates[best_idx]
    mu_b, sig_b = gp.predict(best_x.reshape(1, -1), return_std=True)
    return best_x, float(mu_b.ravel()[0]), float(sig_b.ravel()[0])

# F1 sweep plan -- FINAL coordinate
F1_SWEEP_PLAN = {
    10: np.array([0.227592, 0.561706]),
    11: np.array([0.272742, 0.308863]),
    12: np.array([0.227592, 0.950000]),
    13: np.array([0.591806, 0.239632]),  # CHANGED: W13 -- final sweep point
}

def format_query(x):
    return '-'.join([f'{v:.6f}' for v in np.clip(x, 0.0, 0.999999)])

print('Functions defined. This is the final round.')

## 5. Final Kappa Strategy

**CHANGED from Week 12:** Final round -- all decisions weighted toward preserving best known outputs.

| kappa | Functions | Rationale |
|---|---|---|
| 0.5 | F2, F3, F7 | Exploit near confirmed best regions -- no time to recover from regression |
| 1.0 | F4, F5 | Moderate exploitation toward best known basin |
| 1.5 | F8 | Balanced shot after W12 regression |
| 2.5 | F6 | Final exploratory shot -- best -0.214 never recovered |
| Planned | F1 | Final sweep point [0.591806, 0.239632] |

In [ ]:
# CHANGED: F4 from 2.0->1.0; F5 from 2.0->1.0; F8 from 2.0->1.5 (all final round adjustments)
kappa_per_function = {
    1: None,
    2: 0.5,
    3: 0.5,
    4: 1.0,   # CHANGED from 2.0 -- final shot near R10 basin
    5: 1.0,   # CHANGED from 2.0 -- final moderate exploitation
    6: 2.5,
    7: 0.5,
    8: 1.5,   # CHANGED from 2.0 -- balanced after W12 regression
}

strategy_notes = {
    1: f"FINAL sweep: {format_query(F1_SWEEP_PLAN[13])} -- completes the 4-point systematic grid",
    2: "GP-UCB kappa=0.5 -- 0.665 W12, final exploitation near best 0.678",
    3: "GP-UCB kappa=0.5 -- near best -0.0002, final exploitation",
    4: "GP-UCB kappa=1.0 -- balanced final shot near R10 basin (best 0.520)",
    5: "GP-UCB kappa=1.0 -- moderate final exploitation toward best 6641",
    6: "GP-UCB kappa=2.5 -- F6 never recovered -0.214, final exploratory shot",
    7: "GP-UCB kappa=0.5 -- exploit near R11 all-time best 1.972",
    8: "GP-UCB kappa=1.5 -- balanced after W12 regression from best 9.979",
}
print('Final kappa assignments set.')

## 6. Run Final Week 13 Queries

In [ ]:
CURRENT_ROUND = 13
results = {}

for func_id, d in functions.items():
    X, y = d['X'], d['y']
    n_dims = X.shape[1]
    kappa = kappa_per_function[func_id]
    func_seed = SEED_BASE + func_id

    if func_id == 1:
        best_x = F1_SWEEP_PLAN[CURRENT_ROUND]
        gp = fit_gp(X, y)
        mu_b = float(gp.predict(best_x.reshape(1, -1)).ravel()[0])
        sig_b = float(gp.predict(best_x.reshape(1, -1), return_std=True)[1].ravel()[0])
    else:
        best_x, mu_b, sig_b = suggest_next_query(X, y, n_dims, kappa=kappa, seed=func_seed)

    query_str = format_query(best_x)
    results[func_id] = {
        'query_string': query_str, 'best_x': best_x,
        'current_best_y': float(y.max()), 'predicted_mu': mu_b,
        'predicted_sigma': sig_b, 'kappa': kappa, 'strategy': strategy_notes[func_id],
    }

    print(f"F{func_id} ({n_dims}D) | n={len(y)} | Best Y: {y.max():.6f}")
    print(f"  Strategy : {strategy_notes[func_id]}")
    print(f"  GP mu={mu_b:.4f} | sigma={sig_b:.4f}")
    print(f"  FINAL Query : {query_str}")
    print()

## 7. Final Submission Strings

In [ ]:
print('=' * 65)
print('WEEK 13 -- FINAL SUBMISSION STRINGS')
print('=' * 65)
for func_id, r in results.items():
    print(f'Function {func_id}: {r["query_string"]}')
print('=' * 65)

## 8. Final Cumulative Progress -- All 12 Rounds + Initial

In [ ]:
init_bests = {i: np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy').max() for i in range(1,9)}

rows = []
for i in range(1, 9):
    running = init_bests[i]
    bests = [running]
    for wo in all_week_outputs:
        running = max(running, wo[i])
        bests.append(running)
    improvement = ((bests[-1] - bests[0]) / abs(bests[0]) * 100) if bests[0] != 0 else float('nan')
    rows.append({
        'Fn': f'F{i}', 'Dims': functions[i]['X'].shape[1], 'n': len(functions[i]['y']),
        'Init Best': round(bests[0], 4), 'Best After R12': round(bests[-1], 4),
        'Improvement': f'{improvement:+.1f}%' if not np.isnan(improvement) else 'N/A',
        'kappa W13': kappa_per_function[i]
    })

df = pd.DataFrame(rows).set_index('Fn')
print(df.to_string())

# Visual progress chart
fig, ax = plt.subplots(figsize=(12, 5))
rounds = list(range(13))
labels = ['Init'] + [f'R{r+1}' for r in range(12)]
colors = ['#378ADD','#1D9E75','#D85A30','#BA7517','#7F77DD','#D4537E','#639922','#A32D2D']

for i in range(1, 9):
    running = init_bests[i]
    bests = [running]
    for wo in all_week_outputs:
        running = max(running, wo[i])
        bests.append(running)
    norm_bests = [b / bests[-1] for b in bests]
    ax.plot(labels, norm_bests, marker='o', markersize=4, label=f'F{i}', color=colors[i-1], linewidth=1.5)

ax.set_xlabel('Round')
ax.set_ylabel('Running best (normalized to final best per function)')
ax.set_title('All 8 Functions -- Normalized Progress Across All 12 Query Rounds', fontweight='bold')
ax.legend(ncol=4, fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('final_progress_all_functions.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFinal round complete. 13 weeks, 8 functions, one query per function per week.')